# Dataset rescue statuses

Checks the rescue status of the [example data sources](https://docs.google.com/document/d/1rYsP1I1-TAd3Tu-G6EJ-5o7IbQIHf_dKT3DjH8IpQuo/edit?tab=t.0#heading=h.87di2doy0z55).


## Set up the path


In [1]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

## Load datasets


In [2]:
import pandas as pd

datasets = pd.read_csv("fire_datasets.csv")
datasets

,access_type,name,description,webpage,example_data_url
0,No auth,CarbonPlan Open Climate Risk,Building-level wildfire risk.,https://source.coop/carbonplan/carbonplan-ocr,https://s3.us-west-2.amazonaws.com/us-west-2.o...
1,No auth,cboettig fire,Historical burned-area perimeters and hazard p...,https://source.coop/cboettig/fire,https://data.source.coop/cboettig/fire/usgs-mt...
2,No auth,bkr observations,"Weather station, radiosonde, and aviation obse...",https://source.coop/bkr/obs,NaN
3,No auth,giswqs National Wetlands Inventory,National Wetlands Inventory.,https://source.coop/giswqs/nwi,NaN
4,No auth,USFS probabilistic wildfire risk burn probability,Burn probability and flame-length grids.,https://data-usfs.hub.arcgis.com/datasets/usfs...,https://imagery.geoplatform.gov/iipp/rest/serv...
5,No auth,Wildfire Risk to Communities,Wildfire risk data downloads.,https://wildfirerisk.org/download/,https://wildfirerisk.org/wp-content/uploads/20...
6,No auth,NOAA weather alerts,"Forecasts and active alerts, including Red Fla...",https://www.weather.gov/documentation/services...,https://api.weather.gov/alerts/active?event=Re...
7,No auth,NOAA HMS fire and smoke product,Daily smoke plume and fire detection polygons.,https://www.ospo.noaa.gov/products/land/hms.html,https://satepsanone.nesdis.noaa.gov/pub/FIRE/w...
8,No auth,NOAA Storm Events Database,Historical disaster impacts by county.,https://www.ncdc.noaa.gov/stormevents/,https://www.ncei.noaa.gov/pub/data/swdi/storme...
9,No auth,NASA FIRMS KML fire footprints,Regional active-fire footprint polygons.,https://firms.modaps.eosdis.nasa.gov/,https://firms.modaps.eosdis.nasa.gov/data/acti...


## Check status

Only include the official government data sources that don't require auth.


In [3]:
no_auth = datasets["access_type"] != "Free key"
not_src_coop = ~datasets["webpage"].str.startswith("https://source.coop/")
not_dryad = ~datasets["webpage"].str.contains("/dryad.")
has_example_data_url = datasets["example_data_url"].notna()

datasets_to_check = datasets[no_auth & not_src_coop & not_dryad & has_example_data_url]
datasets_to_check

,access_type,name,description,webpage,example_data_url
4,No auth,USFS probabilistic wildfire risk burn probability,Burn probability and flame-length grids.,https://data-usfs.hub.arcgis.com/datasets/usfs...,https://imagery.geoplatform.gov/iipp/rest/serv...
5,No auth,Wildfire Risk to Communities,Wildfire risk data downloads.,https://wildfirerisk.org/download/,https://wildfirerisk.org/wp-content/uploads/20...
6,No auth,NOAA weather alerts,"Forecasts and active alerts, including Red Fla...",https://www.weather.gov/documentation/services...,https://api.weather.gov/alerts/active?event=Re...
7,No auth,NOAA HMS fire and smoke product,Daily smoke plume and fire detection polygons.,https://www.ospo.noaa.gov/products/land/hms.html,https://satepsanone.nesdis.noaa.gov/pub/FIRE/w...
8,No auth,NOAA Storm Events Database,Historical disaster impacts by county.,https://www.ncdc.noaa.gov/stormevents/,https://www.ncei.noaa.gov/pub/data/swdi/storme...
9,No auth,NASA FIRMS KML fire footprints,Regional active-fire footprint polygons.,https://firms.modaps.eosdis.nasa.gov/,https://firms.modaps.eosdis.nasa.gov/data/acti...
10,No auth,WFIGS / National Interagency Fire Center,Current and historical fire perimeters from US...,https://data-nifc.opendata.arcgis.com/,https://gis.blm.gov/arcgis/rest/services/fire/...
11,No auth,MODIS NDVI,Vegetation-health data useful as a live fuel-c...,https://modis.gsfc.nasa.gov/data/dataprod/mod1...,https://cmr.earthdata.nasa.gov/search/granules...
13,No auth,USDA LANDFIRE,"Wildland fire data, including seasonal fuels.",https://www.landfire.gov/,https://www.landfire.gov/data-downloads/Season...
17,Bulk or analytic,CDC PLACES,Census-tract and county chronic-disease preval...,https://www.cdc.gov/places/,https://data.cdc.gov/resource/i46a-9kgh.geojson


In [4]:
import asyncio

import httpx

from ptps_wildfire_demo.proxy.resolver import Resolver

# essentially an async Series.apply()
timeout = httpx.Timeout(30.0, connect=30.0)
async with httpx.AsyncClient(timeout=timeout) as client:
    resolver = Resolver(client)
    rescues = await asyncio.gather(
        *(resolver.get_rescue(url) for url in datasets_to_check["example_data_url"])
    )


# rescues

In [5]:
rescues_df = pd.DataFrame(rescues).rename(columns={"original_url": "example_data_url"})
results = pd.merge(datasets_to_check, rescues_df, on="example_data_url")

with pd.option_context("display.max_colwidth", 200):
    display(results.drop(columns=["access_type", "description"]))

,name,webpage,example_data_url,wayback_newest_url,drp_metadata_url,drp_download_location
0,USFS probabilistic wildfire risk burn probability,https://data-usfs.hub.arcgis.com/datasets/usfs::probabilistic-wildfire-risk-burn-probability-image-service/explore,"https://imagery.geoplatform.gov/iipp/rest/services/Fire_Aviation/USFS_EDW_RMRS_ProbabilisticWildfireRiskBurnProbability/ImageServer/exportImage?bbox=-2.00375070672E7,2135965.179399997,-7130157.067...",NaN,None,None
1,Wildfire Risk to Communities,https://wildfirerisk.org/download/,https://wildfirerisk.org/wp-content/uploads/2026/04/wrc_download_20260415.xlsx,NaN,None,None
2,NOAA weather alerts,https://www.weather.gov/documentation/services-web-api,https://api.weather.gov/alerts/active?event=Red%20Flag%20Warning,NaN,None,None
3,NOAA HMS fire and smoke product,https://www.ospo.noaa.gov/products/land/hms.html,https://satepsanone.nesdis.noaa.gov/pub/FIRE/web/HMS/Smoke_Polygons/KML/2026/07/hms_smoke20260701.kml,NaN,None,None
4,NOAA Storm Events Database,https://www.ncdc.noaa.gov/stormevents/,https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz,NaN,None,None
5,NASA FIRMS KML fire footprints,https://firms.modaps.eosdis.nasa.gov/,https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip,http://web.archive.org/web/20260110052750/https://firms.modaps.eosdis.nasa.gov/data/active_fire/modis-c6.1/shapes/zips/MODIS_C6_1_USA_contiguous_and_Hawaii_24h.zip,None,None
6,WFIGS / National Interagency Fire Center,https://data-nifc.opendata.arcgis.com/,https://gis.blm.gov/arcgis/rest/services/fire/BLM_Natl_FirePerimeter/MapServer/generateKml,http://web.archive.org/web/20260209203352/https://gis.blm.gov/arcgis/rest/services/fire/BLM_Natl_FirePerimeter/MapServer/generateKml,None,None
7,MODIS NDVI,https://modis.gsfc.nasa.gov/data/dataprod/mod13.php,https://cmr.earthdata.nasa.gov/search/granules.json?short_name=MOD13Q1&page_size=1,NaN,None,None
8,USDA LANDFIRE,https://www.landfire.gov/,https://www.landfire.gov/data-downloads/SeasonalFuels/LF2025_FBFM40_SU26.zip,NaN,None,None
9,CDC PLACES,https://www.cdc.gov/places/,https://data.cdc.gov/resource/i46a-9kgh.geojson,http://web.archive.org/web/20260315024944/https://data.cdc.gov/resource/i46a-9kgh.geojson,None,None
